# HCI2000 Analysis 01: Descriptives and interaction plots

## What this notebook does
The cleaned data are ready. Before running any statistical test, we **look at the data**: what happened in the trials, how the four conditions compare on average,
and what the pattern looks like. This notebook is descriptive only. The statistical tests (normality checks, ANOVAs, the accuracy model) are in `analysis02`.

We produce three things, following Section 4.7 of the paper:

1. **Table A: trial counts per condition.** The paper requires completed and timed-out counts next to every average retrieval time,
   and no-selection counts next to accuracy, because an average over completed trials alone can hide a condition in which many trials failed.
2. **Table B: participant-level values per condition.** One value per participant per condition, summarised across participants. This is the level at which the statistical tests operate.
3. **Interaction plots.** One per outcome. Our primary hypothesis is the orientation x tab-load interaction, which shows up as **non-parallel lines**:
   the gap between horizontal and vertical changes when the load changes.

**Before running this notebook, run `preprocessing.ipynb` first**, because it creates the cleaned files that we read here.

## 0. Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- the ONE line to change if the project folder moves ----
BASE = "/Users/jphonrado/Documents/School/Term 1 A.Y. 26-27/HCI2000/HCI2000-Data-Analysis"

CLEAN_FOLDER = BASE + "/Data Preprocessing/cleaned"      # made by preprocessing.ipynb
TABLE_FOLDER = BASE + "/Data Analysis/tables"             # we save our tables here
FIGURE_FOLDER = BASE + "/Data Analysis/figures"           # and our figures here
os.makedirs(TABLE_FOLDER, exist_ok=True)
os.makedirs(FIGURE_FOLDER, exist_ok=True)

trial = pd.read_csv(CLEAN_FOLDER + "/trial_level.csv")
condition = pd.read_csv(CLEAN_FOLDER + "/condition_level.csv")
print("trial_level:", trial.shape, "  condition_level:", condition.shape)

## 1. Descriptive tables
### Table A: what happened in the trials
For each of the four conditions (orientation x tab load), we count how many trials were completed, timed out, were interrupted, or had no selection at all.
Next to those counts we show accuracy, incorrect selections and tab switches.

Definitions from the paper:
- **Retrieval time and incorrect selections** use **completed** trials.
- **Accuracy** uses every trial where a first selection could be observed. A timeout in which the participant did click something still counts, as an incorrect first selection.
- Completed and timed-out trials are reported **separately** for tab switches.

In [ ]:
# helper columns, so a plain mean uses only the trials we want
trial["incorrect_if_completed"] = trial["incorrect_selections"].where(trial["completed"])
trial["switches_if_completed"] = trial["total_tab_switches"].where(trial["completed"])
trial["switches_if_timed_out"] = trial["total_tab_switches"].where(trial["timed_out"])
trial["no_selection_flag"] = trial["no_selection"] == 1

table_a = trial.groupby(["orientation", "tab_load"]).agg(
    trials=("trial", "count"),
    completed=("completed", "sum"),
    timed_out=("timed_out", "sum"),
    interrupted=("interrupted", "sum"),
    no_selection=("no_selection_flag", "sum"),
    accuracy_attempts=("first_selection_correct", "count"),
    accuracy_percent=("first_selection_correct", "mean"),
    mean_incorrect_selections_completed=("incorrect_if_completed", "mean"),
    mean_tab_switches_completed=("switches_if_completed", "mean"),
    mean_tab_switches_timed_out=("switches_if_timed_out", "mean"),
)
table_a["accuracy_percent"] = table_a["accuracy_percent"] * 100      # proportion -> percent
table_a = table_a.round(2)
table_a.to_csv(TABLE_FOLDER + "/table_A_trial_counts.csv")
table_a

### Table B: what the statistical tests will see
The tests in the paper work with **one value per participant per condition**. So here we take each participant's value in each condition
and summarise those 16 values (or 15, for weighted TLX, because one participant has no weighting form) with a count, a mean and a standard deviation.

In [ ]:
outcomes = ["mean_rt_s", "first_try_accuracy", "weighted_tlx"]

table_b = condition.groupby(["orientation", "tab_load"])[outcomes].agg(["count", "mean", "std"]).round(2)
table_b.to_csv(TABLE_FOLDER + "/table_B_participant_level.csv")
table_b

*Reading Table B:* `count` is the number of participants contributing to that cell. The mean of these participant-level values can differ slightly from Table A,
because Table A pools all trials, while Table B gives every participant equal weight. The tests use Table B.

## 2. Interaction plots
One plot per outcome. Each line is one orientation, the x-axis is the tab load, and each dot is the **average across participants** with an error bar showing **one standard error**
(a measure of how precisely that average is known; shorter bars mean a more precise average).

**How to read them:** if the two lines are parallel, orientation has the same effect at both loads. If the lines **fan out or cross**, the effect of orientation depends on the load.
That non-parallel pattern is exactly what our primary hypothesis (the interaction) is about.

In [ ]:
labels = {"mean_rt_s": "Retrieval time (seconds)",
          "first_try_accuracy": "First-selection accuracy (proportion)",
          "weighted_tlx": "Weighted NASA-TLX (0-100)"}
colors = {"H": "#C8553D", "V": "#2E6F95"}          # the same two colours in every plot
names = {"H": "Horizontal", "V": "Vertical"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, outcome in zip(axes, outcomes):
    # mean and standard error of the 16 participant values, for each orientation x load cell
    means = condition.groupby(["orientation", "tab_load"])[outcome].agg(["mean", "sem"])

    for orientation in ["H", "V"]:
        low = means.loc[(orientation, "low")]
        high = means.loc[(orientation, "high")]
        ax.errorbar([0, 1], [low["mean"], high["mean"]], yerr=[low["sem"], high["sem"]],
                    marker="o", capsize=4, linewidth=2.5, color=colors[orientation], label=names[orientation])

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Low load\n(6 tabs)", "High load\n(16 tabs)"])
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylabel(labels[outcome])
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_FOLDER + "/interaction_plots.png", dpi=200)
plt.show()

**What to say about these plots:** the plots are descriptive. They show the pattern, not whether it is statistically reliable.
Whether the lines are *significantly* non-parallel is what the interaction test in `analysis02` decides. With only 16 participants, we also keep an eye on how long the error bars are.